In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
import pandas as pd
import ast

from scipy import stats
import datetime


from src import paths


from src.utils import interactive_dataframe_selector

In [2]:
# Load environment variables from .env file
load_dotenv("../private_data/.env")

host = os.getenv("HOST")
db = os.getenv("DB")
port = os.getenv("PORT")
role = os.getenv("ROLE")
pw = os.getenv("PASSWORD")
engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

# Creators

In [3]:
query = """
SELECT i.id AS creator_id,
    u.id AS user_id,
    i.name,
    u.email_address,
    u.phone_number,
    u.country_code,
    i.gender,
    CASE WHEN i.birthday < '1900-01-01' THEN NULL ELSE i.birthday END AS birthday,
    i.reviews_count,
    i.rating,
    CASE WHEN u.created_at < '1900-01-01' THEN NULL ELSE u.created_at END AS created_at,
    i.status,
    -- Social Media Aggregation (Similar approach to what works)
    json_agg(
        json_build_object(
            'id', ism.id, 
            'social_media', ism.social_media, 
            'followers', ism.follower, 
            'username', ism.username, 
            'status', ism.status, 
            'reject_reason', ism.reject_reason
        )
    ) FILTER (WHERE ism.id IS NOT NULL) AS socials,
    -- Interest Aggregation (New list of objects or just names)
    json_agg(
        json_build_object(
            'id', ct.id,
            'name', ct.name
        )
    ) FILTER (WHERE ct.id IS NOT NULL) AS interests,
    il.country,
    il.formatted_address,
    il.name AS city,
    il.latitude,
    il.longitude,
    il.place_id
FROM influencers i
JOIN "user" u ON u.id = i.user_id AND u.deleted_at IS NULL
LEFT JOIN images img ON u.images_id = img.id
LEFT JOIN influencer_social_media ism ON ism.influencer_id = i.id AND ism.deleted_at IS NULL
LEFT JOIN influencer_locations il ON i.id = il.influencer_id AND il.deleted_at IS NULL
-- New Joins for Interests
LEFT JOIN influencer_interests ii ON i.id = ii.influencer_id
LEFT JOIN content_types ct ON ii.interest_id = ct.id
WHERE i.deleted_at IS NULL
GROUP BY 
    i.id, u.id, i.name, u.email_address, u.phone_number, u.country_code, 
    i.gender, i.birthday, i.reviews_count, i.rating, img.url, 
    u.created_at, i.status, il.country, il.formatted_address, 
    il.name, il.latitude, il.longitude, il.place_id;
"""
df_creators = pd.read_sql(query, engine)


In [4]:
df_creators['birthday'] = pd.to_datetime(
    df_creators['birthday'], errors='coerce')
df_creators['created_at'] = pd.to_datetime(
    df_creators['created_at'], errors='coerce')

df_creators = df_creators.add_suffix('_creators')

# df_creators.drop(columns=['url'], inplace=True)


# Follower count
# TODO: I can create more finegrained info on the creators here, e.g. to model acceptance rate (maybe this depends on followers across all social media accs.)
def calculate_eligible_followers(x):
    if not isinstance(x, list) or not x:
        return np.nan

    # Extract values, but keep them as None/NaN if missing
    vals = [y.get('followers') for y in x]

    # Filter out None values so we have a clean list of numbers
    clean_vals = [v for v in vals if v is not None]

    # Return max if we have numbers, else return NaN
    return max(clean_vals) if clean_vals else np.nan


df_creators['max_followers_creators'] = df_creators['socials_creators'].apply(
    calculate_eligible_followers)


def flatten_socials(x):
    if not isinstance(x, list):
        return {}
    # Creates {'Instagram': 103, 'TikTok': 50}
    return {item.get('social_media'): item.get('followers') for item in x if item.get('social_media')}


df_creators['socials_map_creators'] = df_creators['socials_creators'].apply(
    flatten_socials)


def standardize_country_names(df, column_name):
    """
    Standardizes a country column by mapping various languages and scripts 
    to a single English name.
    """

    mapping = {
        # Netherlands
        'Hollanda': 'Netherlands', 'Nederland': 'Netherlands', 'Pays-Bas': 'Netherlands',
        'The Netherlands': 'Netherlands', 'Нидерланды': 'Netherlands',
        'เนเธอร์แลนด์': 'Netherlands', '荷兰': 'Netherlands', '荷蘭': 'Netherlands',

        # Germany
        'Alemanha': 'Germany', 'Alemania': 'Germany', 'Almanya': 'Germany',
        'Deutschland': 'Germany', 'Duitsland': 'Germany', 'Germania': 'Germany',
        'Niemcy': 'Germany', 'Германия': 'Germany',

        # Belgium
        'Belgia': 'Belgium', 'Belgien': 'Belgium', 'Belgio': 'Belgium',
        'Belgique': 'Belgium', 'België': 'Belgium', 'Bélgica': 'Belgium',
        'Бельґія': 'Belgium', '比利时': 'Belgium',

        # United Kingdom
        'UK': 'United Kingdom', 'United Kingdom': 'United Kingdom',
        'Reino Unido': 'United Kingdom', 'Unito': 'United Kingdom',
        'Koninkrijk': 'United Kingdom',

        # USA
        'USA': 'United States', 'États-Unis': 'United States',

        # Spain
        'España': 'Spain', 'Spanje': 'Spain',

        # Italy
        'Italien': 'Italy',

        # Austria
        'Oostenrijk': 'Austria',

        # Sweden
        'Suède': 'Sweden',

        # Denmark
        'Dánia': 'Denmark',

        # France
        'Frankrijk': 'France'
    }

    # Use .replace to update the column
    df[column_name] = df[column_name].replace(mapping)

    return df


df_creators = standardize_country_names(df_creators, 'country_creators')


In [22]:
import pandas as pd
import numpy as np
from scipy import stats
import datetime

df = df_creators.copy()

df = df[~df['socials_creators'].isna()]

# Calculate Age
df['birthday_creators'] = pd.to_datetime(df['birthday_creators'], errors='coerce')
current_year = datetime.datetime.now().year
df['age'] = current_year - df['birthday_creators'].dt.year

def get_descriptives(series):
    s = series.dropna()
    if len(s) < 1: 
        return [np.nan] * 5
    mean = np.mean(s)
    sd = np.std(s, ddof=1)
    median = np.median(s)
    q1 = np.percentile(s, 25)
    q3 = np.percentile(s, 75)
    return [mean, sd, median, q1, q3]

# --- 3. BUILD PRESENTATION TABLES ---

# Table 1: Raw Descriptive Stats
metrics = ['age', 'max_followers_creators']
metric_names = ['Age', 'Max Followers']

raw_stats_data = []
for col, name in zip(metrics, metric_names):
    if col in df.columns:
        stats = get_descriptives(df[col])
        raw_stats_data.append([name] + stats)
        
raw_stats = pd.DataFrame(raw_stats_data, columns=['Metric', 'Mean', 'SD', 'Median', '25th Percentile', '75th Percentile'])

# B. Raw Categorical (Counts)
gender_raw = df['gender_creators'].value_counts().reset_index()
gender_raw.insert(0, 'Metric', 'Gender')
gender_raw.columns = ['Metric', 'Value', 'Count']

## Country
# Identify countries with < 10 occurrences and recode them as 'Other'
# We use transform to keep the index aligned with the original dataframe
df['country_creators'] = df['country_creators'].fillna('Unknown')

counts = df.groupby('country_creators')['country_creators'].transform('count')
df['country_raw_grouped'] = df['country_creators'].where(counts >= 10, 'Other')

# Now compute the final counts for your raw table
country_raw = df['country_raw_grouped'].value_counts().reset_index()
country_raw.insert(0, 'Metric', 'Country')
country_raw.columns = ['Metric', 'Value', 'Count']

# Note: If 'Other' already existed from the mapping, 
# value_counts() will automatically sum all 'Other' instances into one row.

raw_demographics = pd.concat([gender_raw, country_raw])

# C. Censored Stats (Percentages & Bins)
bins = [0, 10000, 100000, 1000000, np.inf]
labels = ['Nano (<10k)', 'Micro (10k-100k)', 'Macro (100k-1M)', 'Mega (>1M)']
df['follower_tier'] = pd.cut(df['max_followers_creators'], bins=bins, labels=labels)

# Grouping logic: Percentages for everything
tier_pct = (df['follower_tier'].value_counts(normalize=True) * 100).round(1).reset_index()
gender_pct = (df['gender_creators'].value_counts(normalize=True) * 100).round(1).reset_index()
# country_pct = (df['country_raw_grouped'].value_counts(normalize=True) * 100).round(1).reset_index()

tier_pct.columns = gender_pct.columns = ['Value', 'Percentage (%)']
tier_pct.insert(0, 'Metric', 'Follower Tier')
gender_pct.insert(0, 'Metric', 'Gender')
# country_pct.insert(0, 'Metric', 'Country Distribution')



# --- REFINED CENSORED DEMOGRAPHICS (MCAR Assumption) ---

# 1. Filter out 'Unknown' for the Partner's Geographic Distribution
df_known_geo = df[df['country_raw_grouped'] != 'Unknown'].copy()

# 2. Recalculate Percentages on the KNOWN data only
country_pct = (df_known_geo['country_raw_grouped'].value_counts(normalize=True) * 100).round(1).reset_index()

# 3. Standardize columns
country_pct.columns = ['Value', 'Percentage (%)']
country_pct.insert(0, 'Metric', 'Country (Known Locations)')

# 4. Combine with other metrics (Gender, Follower Tiers)
# Note: You can decide if you want to assume MCAR for Gender as well
censored_data = pd.concat([tier_pct, gender_pct, country_pct], ignore_index=True)


# censored_data = pd.concat([tier_pct, gender_pct, country_pct])



In [16]:
raw_stats = raw_stats.round(1)
raw_stats


,Metric,Mean,SD,Median,25th Percentile,75th Percentile
0,Age,27.6,8.2,26.0,22.0,32.0
1,Max Followers,10728.6,56463.9,1873.0,536.0,6335.0


In [17]:
raw_demographics

,Metric,Value,Count
0,Gender,Female,20476
1,Gender,Male,4137
2,Gender,Other,111
0,Country,Netherlands,13169
1,Country,Unknown,10729
2,Country,Belgium,3509
3,Country,Germany,2952
4,Country,United Kingdom,1430
5,Country,Spain,311
6,Country,United States,118


In [20]:
censored_data

,Metric,Value,Percentage (%)
0,Follower Tier,Nano (<10k),79.8
1,Follower Tier,Micro (10k-100k),18.2
2,Follower Tier,Macro (100k-1M),1.9
3,Follower Tier,Mega (>1M),0.1
4,Gender,Female,82.8
...,...,...,...
70,Country (Known Locations),Switzerland,0.0
71,Country (Known Locations),Hong Kong,0.0
72,Country (Known Locations),Türkiye,0.0
73,Country (Known Locations),Sint Maarten,0.0


In [10]:
df_creators[df_creators.max_followers_creators > 100000]

,creator_id_creators,user_id_creators,name_creators,email_address_creators,phone_number_creators,country_code_creators,gender_creators,birthday_creators,reviews_count_creators,rating_creators,...,socials_creators,interests_creators,country_creators,formatted_address_creators,city_creators,latitude_creators,longitude_creators,place_id_creators,max_followers_creators,socials_map_creators
52,55,65,Emely Beatty,harris.marshall@anonymized.getbarter.com,1-950-959-1685 x21336,31,Male,1995-05-06,0,5.0,...,"[{'id': 56, 'social_media': 'Instagram', 'foll...","[{'id': 26, 'name': 'Gaming'}, {'id': 33, 'nam...",None,None,None,NaN,NaN,None,302000.0,"{'Instagram': 47249, 'Tiktok': 302000}"
74,78,90,Janiya Thompson,braeden.moen@anonymized.getbarter.com,(490) 665-2272 x839,31,Female,2000-03-17,5,5.0,...,"[{'id': 81, 'social_media': 'Tiktok', 'followe...","[{'id': 26, 'name': 'Gaming'}, {'id': 29, 'nam...",Netherlands,"Amsterdam, Netherlands",Amsterdam,52.367573,4.904139,ChIJVXealLU_xkcRja_At0z9AGY,220300.0,"{'Tiktok': 220300, 'Instagram': 7656}"
83,87,99,Craig Wuckert,moore.josie@anonymized.getbarter.com,+15804438809,31,None,NaT,1,5.0,...,"[{'id': 2381, 'social_media': 'Tiktok', 'follo...",None,None,None,None,NaN,NaN,None,750000.0,"{'Tiktok': 750000, 'Instagram': 9806}"
95,101,114,Ms. Anastasia Langosh,ronaldo.lockman@anonymized.getbarter.com,540.329.9176 x9481,31,Female,1991-04-27,0,5.0,...,"[{'id': 106, 'social_media': 'Instagram', 'fol...","[{'id': 23, 'name': 'Food'}, {'id': 29, 'name'...",Netherlands,"Haarlem, Netherlands",Haarlem,52.387388,4.646219,ChIJ--nhYGzvxUcROX6huIBW4Yo,107086.0,"{'Instagram': 107086, 'Tiktok': 13400}"
117,124,137,Waldo Adams,korey.haley@anonymized.getbarter.com,800.410.0089 x66841,31,Male,NaT,0,5.0,...,"[{'id': 131, 'social_media': 'Instagram', 'fol...","[{'id': 22, 'name': 'Travel'}, {'id': 41, 'nam...",Netherlands,"'s-Hertogenbosch, Netherlands",'s-Hertogenbosch,51.697816,5.303675,ChIJN90-rTfuxkcRUHwejVreAAQ,1234868.0,{'Instagram': 1234868}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44221,51179,76760,Ephraim Medhurst III,katelyn.d_amore@anonymized.getbarter.com,+1 (540) 473-2416,31,Female,1991-07-02,0,5.0,...,"[{'id': 49023, 'social_media': 'Tiktok', 'foll...","[{'id': 22, 'name': 'Travel'}, {'id': 31, 'nam...",Netherlands,"Amsterdam, Netherlands",Amsterdam,52.367573,4.904139,ChIJVXealLU_xkcRja_At0z9AGY,264823.0,"{'Tiktok': 72730, 'Instagram': 264823}"
44415,51388,77061,Lauretta Batz I,labadie.kendra@anonymized.getbarter.com,(580) 922-2612,49,Male,2000-03-14,0,5.0,...,"[{'id': 49259, 'social_media': 'Tiktok', 'foll...","[{'id': 21, 'name': 'Fashion'}, {'id': 27, 'na...",Germany,"45 Essen, Germany",Essen,51.457557,7.022463,ChIJOfarlrfCuEcRnSytpBHhAGo,166007.0,{'Tiktok': 166007}
44461,51435,77125,Bridget Dach V,penelope.harris@anonymized.getbarter.com,(440) 224-4607 x824,32,Female,1987-05-21,0,5.0,...,"[{'id': 50074, 'social_media': 'Instagram', 'f...","[{'id': 20, 'name': 'Beauty'}, {'id': 21, 'nam...",Belgium,"Antwerp, Belgium",Antwerp,51.219930,4.414990,ChIJfYjDv472w0cRuIqogoRErz4,208135.0,"{'Instagram': 208135, 'Tiktok': 33566}"
44707,51716,77533,Nayeli Grady,bailey.mckenzie@anonymized.getbarter.com,1-670-477-9273,49,Female,2000-06-17,0,5.0,...,"[{'id': 49594, 'social_media': 'Instagram', 'f...","[{'id': 23, 'name': 'Food'}, {'id': 25, 'name'...",Germany,"72555 Metzingen, Germany",Metzingen,48.539900,9.286406,ChIJhe9SNSLtmUcR9Oec-EBLrOA,130384.0,"{'Instagram': 0, 'Tiktok': 130384}"


In [26]:
# --- 3. EXPORT TO EXCEL ---
with pd.ExcelWriter('../private_data/reports/creator_descriptive_statistics.xlsx', engine='openpyxl') as writer:
    raw_stats.to_excel(writer, sheet_name='Raw_Stats', index=False)
    raw_demographics.to_excel(writer, sheet_name='Raw_Demographics', index=False)
    censored_data.to_excel(writer, sheet_name='Censored_Partner_View', index=False)

# --- 3. EXPORT TO EXCEL ---
with pd.ExcelWriter('../private_data/reports/creator_descriptive_statistics_censored.xlsx', engine='openpyxl') as writer:
    censored_data.to_excel(writer, sheet_name='Censored_Partner_View', index=False)



print("Export Complete: 'creator_descriptive_statistics.xlsx' is ready for Slack/Email.")

Export Complete: 'creator_descriptive_statistics.xlsx' is ready for Slack/Email.
